### DB日報資料撈取

In [3]:
import pandas as pd
import numpy as np
import pyodbc
import configparser

In [ ]:
cfg = configparser.ConfigParser()
cfg.read(r"D:\yujui\SqlServer18.txt")      # ← 指定絕對路徑
print(cfg.sections())                       # 應該會看到 ['mssql']

if 'mssql' not in cfg:
    raise RuntimeError("找不到 [mssql] 區段，請檢查 ini 檔位置與內容")

sec = cfg['mssql']
database = "DSC_CRM_SP"   # 寫在程式內

conn = pyodbc.connect(
    f"DRIVER={{SQL Server}};"
    f"SERVER={sec['server']};DATABASE={database};"
    f"UID={sec['uid']};PWD={sec['pwd']};",
    autocommit=True
)

['mssql']


In [ ]:
query = """
SELECT GY001 AS LD005, GY012 AS LD010
FROM CRMGY
WHERE LEN(GY012) >= 5
"""
LeadInfo = pd.read_sql(query, conn)
lead_df = LeadInfo[["LD005", "LD010"]].copy()
lead_df["LD010"] = lead_df["LD010"].fillna("").astype(str)
print(f"總筆數：{len(lead_df):,}")
lead_df.head(3)

C:\Users\DSC\AppData\Local\Temp\ipykernel_27352\3919206088.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  LeadInfo = pd.read_sql(query, conn)


,LD005,LD010
0,0000000001,致電03 452 2102，電話已是空號..查無其他有效電話...且google顯示暫停營業
1,0000000016,當初導入失敗，user認為鼎新系統卡控太多，\n目前為淡季生意很差，所以也不會購買任何系統
2,0000000016,OT探詢
3,0000000016,能管探詢
4,0000000017,確認ERP與異質系統整合需求
...,...,...
756984,Y000000000,回撥線索。手機一樣是暫停使用，mail也無回應@@故先結案處理。
756985,Y000000000,軍方走招標
756986,Y000000000,9/17 16:27去電曾R，掛斷電話轉語音信箱。
756987,Y000000000,2025-12-22 11:04無人接聽


In [4]:
import os, configparser
import google.generativeai as genai

cfg = configparser.ConfigParser()
cfg.read(r"D:\yujui\GoogleCloud.ini")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", cfg.get("gcp", "gemini_api_key", fallback=""))
if not GEMINI_API_KEY:
    raise RuntimeError("找不到 gemini_api_key，請確認 D:\\yujui\\GoogleCloud.ini")

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")
print("✅ Gemini API 設定完成")


d:\miniforge3\envs\yujui_even\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
d:\miniforge3\envs\yujui_even\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DSC\AppData\Local\Temp\ipykernel_23948\3795072751.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://gith

In [9]:
import os, re, json, csv, time
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# ════════════════════════════════════════════════════════════════
# 關鍵詞快篩字典（Stage A > B > C1 > C2 > D > E > none）
# ════════════════════════════════════════════════════════════════
_KW = {
    "A": ["簽約","合約","議價","議約","成交","用印","簽署","下訂",
          "訂單確認","定案","採購單","付款","發票","尾款","首款",
          "簽訂","正式合約","合約簽訂","簽合約","已簽"],
    "B": ["DEMO","demo","展示","報價","簡報","競爭","POC","poc",
          "評估方案","競標","提案報告","技術評估","排除疑慮","決標",
          "比稿","投標","標案","比較方案","方案評估","報價單"],
    "C1": ["立案","決策者","高層拜訪","形成共識","採購委員會",
           "需求確認","RFP","規格書","評估名單","專案啟動",
           "納入評估","正式評估","列入評估","提需求","需求書"],
    "C2": ["痛點","問題點","探詢需求","認同問題","初步提案",
           "商機確認","提出解決","客戶認同","擴大痛點","需求探討",
           "痛點確認","找到問題","問題確認","提出方向"],
    "D":  ["初訪","潛在客戶","引發興趣","初步接觸","有興趣了解",
           "開發拜訪","感興趣","有意願了解","想了解","初步了解",
           "初次拜訪","首次拜訪","建立關係","開發客戶"],
    "E":  ["潛客","電話開發","名單","陌開","冷開","陌生開發"],
}

_NONE_KW = ["電話未接","未接電話","不在位","無人接聽","已離職","婉拒",
            "拒絕","無需求","暫無需求","沒有興趣","已有系統","已導入",
            "無法聯繫","關機","空號","電話打不通","失聯"]

_STAGE_GROUP = {"A":"成交後","B":"成交前","C1":"成交前",
                "C2":"成交前","D":"成交前","E":"成交前"}

_stats = {"kw": 0, "llm": 0}

# ── 基礎結果 ─────────────────────────────────────────────────────
def _none_result(reason=""):
    return {"E":False,"D":False,"C2":False,"C1":False,
            "B":False,"A":False,"stage_group":"none","reason":reason}

def _stage_result(stage, reason):
    stage = str(stage)  # 確保是 Python str（KNN 回傳 np.str_）
    if stage not in _STAGE_GROUP:   # "none" 或未知 stage
        return _none_result(reason)
    r = _none_result(reason)
    r[stage] = True
    r["stage_group"] = _STAGE_GROUP[stage]
    return r

# ── 關鍵詞快篩 ───────────────────────────────────────────────────
def keyword_stage_check(text):
    t = text or ""
    if len(t.strip()) < 15:
        return _none_result("文字過短")
    for kw in _NONE_KW:
        if kw in t:
            return _none_result(kw)
    for stage in ["A","B","C1","C2","D","E"]:
        for kw in _KW[stage]:
            if kw in t:
                return _stage_result(stage, kw)
    return None

# ── 批次 LLM（10 筆/call，節省 RPM）────────────────────────────
BATCH_LLM_N = 10

_BATCH_PROMPT = (
    "以下 {n} 筆業務日誌，逐筆判斷銷售階段。\n"
    "階段：A議價簽約 / B報價DEMO評估 / C1立案決策者 / C2確認痛點 / D引發興趣 / E陌生開發 / none無明確行為\n\n"
    "每筆輸出一行，格式「編號|代碼|關鍵依據15字內」，不加其他文字：\n\n"
    "{records}"
)

def _parse_batch_line(line, n):
    parts = line.split("|")
    if len(parts) < 2:
        return None, None
    try:
        idx = int(re.sub(r"\D", "", parts[0])) - 1
    except ValueError:
        return None, None
    if not (0 <= idx < n):
        return None, None
    stage_raw = parts[1].strip().upper()
    reason = parts[2].strip()[:30] if len(parts) > 2 else ""
    for s in ["A","B","C1","C2","D","E"]:
        if s in stage_raw:
            return idx, _stage_result(s, reason)
    return idx, _none_result(reason or "無明確銷售行為")

def llm_batch_check(texts, max_retries=3):
    n = len(texts)
    records = "\n".join(f"[{i+1}] {t[:400]}" for i, t in enumerate(texts))
    prompt = _BATCH_PROMPT.format(n=n, records=records)
    for attempt in range(max_retries):
        try:
            resp = model.generate_content(prompt, request_options={"timeout": 60})
            results = [None] * n
            for line in resp.text.strip().splitlines():
                line = line.strip()
                if not line:
                    continue
                idx, res = _parse_batch_line(line, n)
                if idx is not None:
                    results[idx] = res
            return [r if r is not None else _none_result("批次解析失敗") for r in results]
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return [_none_result("逾時/失敗")] * n

# ── 並行化：關鍵詞全量快篩 → LLM 批次補完 ───────────────────────
def parallel_stage_check(batch, max_workers=6):
    records = batch.to_dict("records")
    results = [None] * len(records)
    llm_indices = []

    for i, row in enumerate(records):
        kw = keyword_stage_check(row["LD010"])
        if kw is not None:
            results[i] = kw
            _stats["kw"] += 1
        else:
            llm_indices.append(i)

    if llm_indices:
        _stats["llm"] += len(llm_indices)
        llm_texts = [records[i]["LD010"] for i in llm_indices]
        mini_batches = [llm_texts[i:i+BATCH_LLM_N]
                        for i in range(0, len(llm_texts), BATCH_LLM_N)]
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            batch_results = list(ex.map(llm_batch_check, mini_batches))
        flat = [r for sub in batch_results for r in sub]
        for i, idx in enumerate(llm_indices):
            results[idx] = flat[i]

    return pd.DataFrame(results, index=batch.index)

print(f"✅ 基礎函式載入完成")

# ════════════════════════════════════════════════════════════════
# KNN 分類（TF-IDF + sklearn，零 API 呼叫）
# ════════════════════════════════════════════════════════════════
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import pickle
from pathlib import Path

KNN_K          = 5
KNN_CONF       = 0.65   # ≥ 4/5 票視為高信心（同 L1L7 Step2）
KNN_MODEL_PATH = r"D:\yujui\痛點需求地圖\stage_knn_model.pkl"

knn        = None   # 訓練完再 assign
vectorizer = None   # TF-IDF vectorizer

# 若模型已存在則直接載入
if Path(KNN_MODEL_PATH).exists():
    _m = pickle.load(open(KNN_MODEL_PATH, "rb"))
    knn = _m["knn"]
    vectorizer = _m.get("vectorizer")
    print(f"✅ 已載入既有 KNN 模型（{_m.get('n_train',0):,} 筆種子）")

def batch_embed(texts):
    """TF-IDF 向量化，回傳 sparse matrix（零 API 呼叫）"""
    return vectorizer.transform([t[:800] for t in texts])

# ── 三層 Pipeline：關鍵詞 → TF-IDF KNN → LLM batch ──────────────
def parallel_stage_check_knn(batch, max_workers=6):
    records = batch.to_dict("records")
    results = [None] * len(records)
    knn_indices = []
    llm_indices = []

    # Layer 1：關鍵詞快篩
    for i, row in enumerate(records):
        kw = keyword_stage_check(row["LD010"])
        if kw is not None:
            results[i] = kw
            _stats["kw"] += 1
        else:
            knn_indices.append(i)

    # Layer 2：TF-IDF KNN（信心 >= KNN_CONF 直接給結果）
    if knn is not None and vectorizer is not None and knn_indices:
        knn_texts = [records[i]["LD010"] for i in knn_indices]
        X = batch_embed(knn_texts)
        proba = knn.predict_proba(X)
        classes = knn.classes_
        for j, i in enumerate(knn_indices):
            conf = float(proba[j].max())
            stage = str(classes[proba[j].argmax()])
            if conf >= KNN_CONF:
                results[i] = _stage_result(stage, f"KNN {conf:.0%}")
            else:
                llm_indices.append(i)
    else:
        llm_indices = knn_indices   # 無模型時全走 LLM

    # Layer 3：LLM batch（低信心剩餘，約 5%）
    if llm_indices:
        _stats["llm"] += len(llm_indices)
        llm_texts = [records[i]["LD010"] for i in llm_indices]
        mini_batches = [llm_texts[k:k+BATCH_LLM_N]
                        for k in range(0, len(llm_texts), BATCH_LLM_N)]
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            batch_results_llm = list(ex.map(llm_batch_check, mini_batches))
        flat = [r for sub in batch_results_llm for r in sub]
        for j, i in enumerate(llm_indices):
            results[i] = flat[j]

    return pd.DataFrame(results, index=batch.index)

print(f"✅ KNN 函式載入完成（TF-IDF，信心門檻 {KNN_CONF:.0%}）")
print("✅ parallel_stage_check_knn 已定義（三層：關鍵詞→KNN→LLM）")


## 主流程：全量商機等級判定（756,989 筆）

斷點續傳 + CSV 輸出，不依賴 BigQuery。

In [ ]:
import json, time, csv as csv_mod
from pathlib import Path

OUTPUT_FILE   = r"D:\yujui\痛點需求地圖\lead_stage_results.csv"
PROGRESS_FILE = r"D:\yujui\痛點需求地圖\lead_stage_progress.json"

def get_last_progress() -> int:
    if Path(PROGRESS_FILE).exists():
        return json.loads(Path(PROGRESS_FILE).read_text()).get("last_start", 0)
    return 0

def save_progress(start: int):
    Path(PROGRESS_FILE).write_text(json.dumps({"last_start": start}))

def apply_decision_rules(df: pd.DataFrame) -> pd.DataFrame:
    """每筆日誌取最高銷售階段作為代表（A > B > C1 > C2 > D > E > none）"""
    stage_order = ["A", "B", "C1", "C2", "D", "E"]
    def top_stage(row):
        for s in stage_order:
            if row.get(s, False):
                return s
        return "none"
    df = df.copy()
    df["top_stage"] = df.apply(top_stage, axis=1)
    return df

print("✅ 輔助函式載入完成")
print(f"   輸出檔：{OUTPUT_FILE}")
print(f"   進度檔：{PROGRESS_FILE}")

In [ ]:
# ── 建立 KNN 分類器（用現有標注結果當種子）──────────────────────
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize
import pickle

VALID_STAGES = ["A","B","C1","C2","D","E","none"]
MIN_PER_CLASS = 5   # 每個 stage 至少需要幾筆才納入訓練

seed_df = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig", low_memory=False)
seed_df = seed_df[seed_df["top_stage"].isin(VALID_STAGES)].dropna(subset=["LD010","top_stage"])
print(f"種子資料：{len(seed_df):,} 筆")
print(seed_df["top_stage"].value_counts().to_string())

# 各 stage 取樣上限（避免 none 嚴重失衡）
MAX_PER_CLASS = 3000
train_parts = []
for stage, grp in seed_df.groupby("top_stage"):
    if len(grp) >= MIN_PER_CLASS:
        train_parts.append(grp.sample(min(len(grp), MAX_PER_CLASS), random_state=42))
train_df = pd.concat(train_parts).sample(frac=1, random_state=42)
print(f"\n訓練集：{len(train_df):,} 筆（各類上限 {MAX_PER_CLASS}）")

# TF-IDF 向量化（字元 2-3 gram，無需 API）
print("\n訓練 TF-IDF 向量化器...")
train_texts  = train_df["LD010"].astype(str).tolist()
train_labels = train_df["top_stage"].tolist()

vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 3),
    max_features=30000, sublinear_tf=True
)
X_train = vectorizer.fit_transform(train_texts)
print(f"  特徵維度：{X_train.shape[1]:,}")

# 訓練 KNN
knn = KNeighborsClassifier(n_neighbors=KNN_K, metric="cosine", algorithm="brute", n_jobs=-1)
knn.fit(X_train, train_labels)
print(f"\n✅ KNN 訓練完成（k={KNN_K}，{len(train_labels)} 筆種子，零 API 呼叫）")

# 儲存
with open(KNN_MODEL_PATH, "wb") as f:
    pickle.dump({"knn": knn, "vectorizer": vectorizer, "n_train": len(train_labels)}, f)
print(f"✅ 模型已儲存：{KNN_MODEL_PATH}")


In [ ]:
# ────────────────────────────────────────────────────────────────
# 主流程：全量商機等級判定（756,989 筆），斷點續傳，輸出 CSV
# ────────────────────────────────────────────────────────────────
CHUNK_SIZE  = 5000   # 每批 5000 筆（保守，避免 OOM）
MAX_WORKERS = 6      # ThreadPoolExecutor 並發數

total       = len(lead_df)
start_point = get_last_progress()
file_exists = Path(OUTPUT_FILE).exists() and start_point > 0

print(f"總筆數：{total:,}  |  從第 {start_point:,} 筆繼續  |  預計批次：{(total - start_point) // CHUNK_SIZE + 1}")

for start in tqdm(range(start_point, total, CHUNK_SIZE), desc="商機等級", unit="批", ncols=100):
    batch = lead_df.iloc[start : start + CHUNK_SIZE].copy()
    batch["LD010"] = batch["LD010"].astype(str).str.replace("\n", " ", regex=False).str.replace("\r", " ", regex=False)

    # LLM 並行判斷
    batch_results = parallel_stage_check_knn(batch, max_workers=MAX_WORKERS)
    batch = pd.concat([batch, batch_results], axis=1)

    # 取最高階段
    batch = apply_decision_rules(batch)

    # 追加寫入 CSV
    batch.to_csv(
        OUTPUT_FILE,
        mode="a",
        index=False,
        header=not file_exists,
        encoding="utf-8-sig",
        quoting=csv_mod.QUOTE_ALL,
    )
    file_exists = True

    # 存進度（下次從 start + CHUNK_SIZE 繼續）
    save_progress(start + CHUNK_SIZE)

    time.sleep(0.5)  # 輕微緩衝，避免 RPM 觸頂

print(f"\n✅ 全量完成！結果存於：{OUTPUT_FILE}")

# 快速統計
result_df = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
print(f"\n📊 top_stage 分布（共 {len(result_df):,} 筆）：")
print(result_df["top_stage"].value_counts().to_string())